In [1]:
import numpy as np
import lib_debug

%load_ext autoreload
%autoreload 2

In [ ]:
# -------------------- robust LCB on J(x) Bayesian Optimization --------------------
# This notebook is intentionally minimal: it only runs robust LCB on J(x).
# sEI / DGSM / StableOpt / AIRBO / uGP-UCB / USeMO / Unscented BO / CVaR /
# mean-variance robust objectives are not used here.

# -------------------- 1. Synthetic function switch --------------------
# Switch the synthetic function by comment-in / comment-out.
# Keep the variable names below unchanged when changing problems.

function_name = "branin"
d = 2
gamma = 0.02

# function_name = "hartmann6"
# d = 6
# gamma = 1.0

# function_name = "ackley"
# d = 6
# gamma = 0.05

f_true, bounds, Sigma, canonical_name = lib_debug.get_synthetic_problem(
    function_name=function_name,
    d=d,
)
lower_bounds = bounds[:, 0]
upper_bounds = bounds[:, 1]

# -------------------- 2. Default experiment settings --------------------
random_seed = 0
rng = np.random.default_rng(random_seed)
noise_std = 0.0
noise_var = noise_std ** 2
n_initial = 5 * d
n_iter = 30
n_perturb_acq = 64
n_perturb_validation = 512
kappa = 2.0
n_candidates = 2000 if d <= 2 else 5000

STRATEGY = "robust_lcb_J"
print(f"--- Strategy Selected: {STRATEGY} ---")
print(f"--- Function Selected: {canonical_name} (d={d}) ---")

# -------------------- 3. Initialization --------------------
X_train = rng.uniform(lower_bounds, upper_bounds, size=(n_initial, d))
y_train = f_true(X_train).reshape(-1, 1)
if noise_std > 0:
    y_train = y_train + rng.normal(0.0, noise_std, size=y_train.shape)

# -------------------- 4. Bayesian Optimization Loop --------------------
print(f"--- Starting Optimization Loop ({n_iter} iterations) ---")
print(f"{'Iter':<5} | {'Best y':<12} | {'New y':<12} | {'robust LCB J':<14}")
print("-" * 60)

x_robust_lcb_J = None
for i in range(n_iter):
    # 1. Fit GP to the current nominal observations D={(X, y)}.
    gp = lib_debug.KernelGPRegressor(
        kernel=lib_debug.rbf_kernel,
        gamma=gamma,
        noise_var=noise_var,
    ).fit(X_train, y_train)

    # 2. Minimize alpha_rLCB(x)=mu_J(x)-kappa*sigma_J(x).
    #    The optimizer uses common random perturbation deltas within this BO iteration.
    x_next, robust_lcb_value = lib_debug.optimize_robust_lcb_by_random_search(
        gp,
        Sigma,
        bounds,
        rng,
        n_perturb=n_perturb_acq,
        kappa=kappa,
        n_candidates=n_candidates,
    )

    # 3. Observe the nominal black-box function once at x_next.
    #    We do not observe f_true(x_next + delta) in this PoC.
    y_next = f_true(x_next.reshape(1, -1)).reshape(1, 1)
    if noise_std > 0:
        y_next = y_next + rng.normal(0.0, noise_std, size=(1, 1))

    # 4. Update data.
    X_train = np.vstack([X_train, x_next])
    y_train = np.vstack([y_train, y_next])
    x_robust_lcb_J = x_next

    print(
        f"{i + 1:<5} | {np.min(y_train):<12.6f} | "
        f"{y_next.item():<12.6f} | {robust_lcb_value:<14.6f}"
    )

# -------------------- 5. Final surrogate robust mean validation --------------------
gp = lib_debug.KernelGPRegressor(
    kernel=lib_debug.rbf_kernel,
    gamma=gamma,
    noise_var=noise_var,
).fit(X_train, y_train)

best_idx_final = int(np.argmin(y_train))
best_observed_x = X_train[best_idx_final]
best_observed_y = float(y_train[best_idx_final, 0])

validation = lib_debug.validate_surrogate_robust_mean(
    {
        "best_observed": best_observed_x,
        "robust_lcb_J": x_robust_lcb_J,
    },
    gp,
    Sigma,
    bounds,
    n_mc=n_perturb_validation,
    rng=rng,
    use_cov=True,
)
robust_best = min(validation, key=lambda name: validation[name]["posterior_robust_mean"])

# -------------------- 6. Required summaries --------------------
print("\nBO summary")
print(f"function name: {canonical_name}")
print(f"dimension d: {d}")
print(f"bounds:\n{bounds}")
print(f"n_initial: {n_initial}")
print(f"n_iter: {n_iter}")
print(f"n_perturb_acq: {n_perturb_acq}")
print(f"kappa: {kappa}")
print(f"Sigma:\n{Sigma}")
print(f"best observed y: {best_observed_y:.6f}")
print(f"best observed x: {best_observed_x}")
print(f"final robust_lcb_J recommended x: {x_robust_lcb_J}")

print("\nvalidation summary")
header = (
    f"{'candidate':<18} {'nominal_posterior_mean':>24} "
    f"{'nominal_posterior_std':>23} {'posterior_robust_mean':>24} "
    f"{'posterior_robust_std':>23}"
)
print(header)
print("-" * len(header))
for candidate, stats in validation.items():
    print(
        f"{candidate:<18} {stats['nominal_posterior_mean']:>24.6f} "
        f"{stats['nominal_posterior_std']:>23.6f} "
        f"{stats['posterior_robust_mean']:>24.6f} "
        f"{stats['posterior_robust_std']:>23.6f}"
    )
print(f"\nrobust best by posterior_robust_mean: {robust_best}")

